# 2. Ringing a black hole and extracting gravitational waves

A Schwarzschild black hole ($M=1$) in Kerr-Schild coordinates, with an $\ell=2, m=0$ perturbation added to the metric.
We compute $\Psi_4$, decompose it into spin-weighted harmonics on a geodesic sphere, and read it with kuibit.

The physics target is the quasi-normal mode $M\omega_{220} = 0.3737 - 0.0890\,i$. As you will see, the plain ADM system
becomes unstable before a clean ringdown can be measured. Finding out *why* is the point of this notebook, and it
motivates BSSN.

The full run (`par/schwarzschild_perturbed.par`) takes ~15 min on 12 cores. Below we shorten it to $t = 24$.

In [ ]:
from pynr import Simulation
import numpy as np, matplotlib.pyplot as plt
over = {"IO::out_dir": "bh_small", "IOBasic::outInfo_every": 40, "Cactus::cctk_final_time": 24.0}
sim = Simulation.from_parfile("../par/schwarzschild_perturbed.par", overrides=over).run()

In [ ]:
from kuibit.simdir import SimDir
sd = SimDir("bh_small")
print("radii:", sd.gws.radii)
r = max(sd.gws.radii)
psi4 = sd.gws[r][(2, 0)]
plt.semilogy(psi4.t, np.abs(psi4.y))
plt.xlabel("t / M"); plt.ylabel(r"$|r\Psi_4^{20}|$ (not rescaled)")

## Checks and a (failing) ringdown fit

1. Symmetry: $\operatorname{Im}\Psi_4^{20}$ and the odd-$\ell$ modes should be at round-off level. Why?
2. Try to fit $A e^{-t/\tau}\cos(\omega t + \phi)$ after the burst and compare with $\omega = 0.3737$, $\tau = 11.2$.
   Then plot `sd.ts.norm2["H"]` on a log scale. When does the constraint violation start to grow, and how does
   that time compare with the QNM period?

In [ ]:
from scipy.optimize import curve_fit
t0, t1 = 12.0, 24.0      # adjust after looking at the plot above
m = (psi4.t > t0) & (psi4.t < t1)
f = lambda t, A, tau, w, ph: A * np.exp(-(t - t0) / tau) * np.cos(w * (t - t0) + ph)
p, _ = curve_fit(f, psi4.t[m], psi4.y.real[m], p0=[np.abs(psi4.y[m]).max(), 11, 0.37, 0])
print(f"omega = {p[2]:.4f}  (QNM 0.3737),  tau = {p[1]:.2f}  (QNM 11.24)")
plt.plot(psi4.t, psi4.y.real); plt.plot(psi4.t[m], f(psi4.t[m], *p), "--")

### Things to try
* Compare the $(2,0)$ and $(4,0)$ modes as functions of time. Which one grows, and why is that a sign of a numerical
  rather than a physical mode?
* Change `Dissipation::epsdis`, `ADMEvolve::excision_radius` and the resolution. How do they affect the time at which $H$ starts growing?
* The perturbation does not satisfy the Hamiltonian constraint. How large is $H$ at $t=0$ compared with the unperturbed run?